# Qwen2.5-Omni Attention Extraction & Region Analysis Pipeline

This notebook extracts **audio → image cross-attention** from Qwen2.5-Omni-7B, then analyzes the attention distribution across four image quadrants (TL/TR/BL/BR).

**Workflow:**
1. Configure all paths in the cell below
2. Download & load the model
3. Extract per-layer attention matrices from each (audio, image) pair → save as `.pkl`
4. Analyze PKL files: map attention to image quadrants → export as `.csv`

## 1. User Configuration

**Please fill in all paths below before running.**

In [ ]:
# [User Configuration]
# Set all paths, select AUDIO_MODE, and adjust parameters here.
# No output files. This cell must be run before all other cells.
# To switch audio mode, change AUDIO_MODE and re-run this cell.
# ==============================================================================
# USER CONFIGURATION - Please modify all paths below
# ==============================================================================
from pathlib import Path

# --- Model paths ---
# Directory for HuggingFace cache (model weights will be cached here)
HF_HOME = Path("/path/to/hf_home")
# Local directory to store the downloaded Qwen2.5-Omni-7B model
MODEL_DIR = Path("/path/to/hf_models/Qwen2.5-Omni-7B")

# --- Data root ---
# Root directory containing your .xlsx files
ROOT_DIR = Path("/path/to/your/project_root")

# --- Image directory (shared across all audio modes) ---
BASE_IMAGE_DIR_STR = "/path/to/your/image_directory"

# --- Audio mode selection ---
# Choose ONE of: "before_tar", "before_er", "before_sp"
# This determines which audio directory and output paths are used
# for BOTH attention extraction and region analysis.
AUDIO_MODE = "before_tar"  # <-- change this to switch mode

# Base directory that contains the three audio subdirectories
BASE_AUDIO_ROOT = "/path/to/your/audio_root"

# Mode config: mode name -> (audio_dir, pkl_subdir, csv_subdir)
# Audio dir is under BASE_AUDIO_ROOT; pkl/csv subdirs are under ROOT_DIR.
# You can edit subdirectory names if your folder structure differs.
_AUDIO_MODE_CONFIG = {
    "before_tar": {
        "audio_dir": BASE_AUDIO_ROOT + "/audio_cut_before_tar",
        "pkl_subdir": "audio_cut_before_tar/attention_pkls_raw",
        "csv_subdir": "audio_cut_before_tar/quad_outputs_fixedboxes",
    },
    "before_er": {
        "audio_dir": BASE_AUDIO_ROOT + "/audio_cut_before_er",
        "pkl_subdir": "audio_cut_before_er/attention_pkls_raw",
        "csv_subdir": "audio_cut_before_er/quad_outputs_fixedboxes",
    },
    "before_sp": {
        "audio_dir": BASE_AUDIO_ROOT + "/audio_cut_before_sp",
        "pkl_subdir": "audio_cut_before_sp/attention_pkls_raw",
        "csv_subdir": "audio_cut_before_sp/quad_outputs_fixedboxes",
    },
}

assert AUDIO_MODE in _AUDIO_MODE_CONFIG, \
    f"Invalid AUDIO_MODE '{AUDIO_MODE}'. Choose from: {list(_AUDIO_MODE_CONFIG.keys())}"
ACTIVE_CONDITION = _AUDIO_MODE_CONFIG[AUDIO_MODE]

# --- Canvas & Region settings ---
# Canvas size of your stimulus images (pixels)
CANVAS_W, CANVAS_H = 1008, 756

# Four quadrant regions [x1, y1, x2, y2] in pixel coordinates
REGIONS = {
    "TL": [84, 28, 392, 336],
    "TR": [616, 28, 924, 336],
    "BL": [84, 420, 392, 728],
    "BR": [616, 420, 924, 728],
}

# --- Attention extraction settings ---
HEAD_AGG = "mean"          # Head aggregation: 'mean' or 'max'
DTYPE_SAVE = "float16"     # Save precision: 'float16' or 'float32'
AUDIO_STRIDE = 1           # Audio token downsampling (1 = no downsampling)
MAX_AUDIO_TOKENS = None    # Max audio tokens (None = no limit)
LAYER_STRIDE = 1           # Layer downsampling (1 = all layers)

# --- Region analysis settings ---
REGION_AGG = "mean"        # Region aggregation: 'mean', 'sum', or 'max'
DECIMALS = 6               # Decimal places in CSV output
INCLUDE_REST = True        # True: output TL/TR/BL/BR/REST (5 regions); False: 4 regions only
FAIL_FAST = True           # Stop on first error in batch processing

# --- ViT patch size (Qwen-VL series typically uses 14) ---
PATCH_SIZE = 14

# --- Excel sheet name (None = first sheet) ---
SHEET_NAME = None

# --- Prompt template ---
PROMPT_TEMPLATE = (
"""## 指令\n你是一个心理语言学实验的被试。接下来，你会首先看到四幅图，然后会听到一个和图片相关的句子（句子中会提到某些图片）。你的任务是：仔细看图片，并认真听句子，理解其意思。注意：某些试次结束时，会有一个关于本试次的问题。以下是图片和句子：<image><audio> """
)

print(f"Configuration loaded. AUDIO_MODE = '{AUDIO_MODE}'")
print(f"  Audio dir:  {ACTIVE_CONDITION['audio_dir']}")
print(f"  PKL output: {ROOT_DIR / ACTIVE_CONDITION['pkl_subdir']}")
print(f"  CSV output: {ROOT_DIR / ACTIVE_CONDITION['csv_subdir']}")

## 2. Install Dependencies

In [ ]:
# [Install Dependencies]
# Installs/upgrades required Python packages.
# No output files. You may need to restart the runtime after first install.
!pip install -U "transformers>=4.57" torchcodec torchaudio openpyxl

## 3. Imports

In [ ]:
# [Import Libraries]
# Imports all required Python modules.
# No output files. If any import fails, check that the dependencies cell above was run.
import os
import math
import pickle
from pathlib import Path
from typing import Optional, List, Dict, Tuple

import pandas as pd
import torch
import torchaudio
import numpy as np
from PIL import Image
from transformers import (
    Qwen2_5OmniThinkerForConditionalGeneration,
    Qwen2_5OmniProcessor,
)
from huggingface_hub import snapshot_download

print("All imports successful.")

## 4. Download & Load Model

In [ ]:
# [Download Model]
# Creates necessary directories and downloads Qwen2.5-Omni-7B model files
# from HuggingFace Hub to MODEL_DIR. Skips files that already exist.
# Output: model weights, config, and tokenizer files saved to MODEL_DIR (~15GB on first run).
HF_HOME.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
ROOT_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HOME / "hub")

repo_id = "Qwen/Qwen2.5-Omni-7B"
print(f"Downloading {repo_id} to {MODEL_DIR} ...")
snapshot_download(
    repo_id=repo_id,
    local_dir=str(MODEL_DIR),
    local_dir_use_symlinks=False,
    resume_download=True,
    ignore_patterns=[],
    allow_patterns=["*.json", "*.safetensors", "*.py", "*.md", "token*", "*.txt"],
)
print("Model downloaded to:", MODEL_DIR)

In [ ]:
# [Load Model & Processor]
# Loads the Qwen2.5-Omni-7B model and processor into GPU/CPU memory.
# Uses attn_implementation="eager" so that output_attentions works correctly.
# No output files. This cell takes a few minutes on first load.
MODEL_ID_OR_PATH = str(MODEL_DIR)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


def load_model_and_processor():
    """
    Load Qwen2.5-Omni-7B with attn_implementation='eager'
    so that output_attentions is available.
    """
    if torch.cuda.is_available():
        major, _ = torch.cuda.get_device_capability()
        dtype = torch.bfloat16 if major >= 8 else torch.float16
        device_map = "auto"
    else:
        dtype = torch.float32
        device_map = None

    print(f"Loading model from {MODEL_ID_OR_PATH} (attn_implementation='eager') ...")
    model = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(
        MODEL_ID_OR_PATH,
        torch_dtype=dtype,
        device_map=device_map,
        low_cpu_mem_usage=True,
        attn_implementation="eager",
    )
    if device_map is None:
        model = model.to(DEVICE)
    model.eval()

    print(f"Loading processor from {MODEL_ID_OR_PATH} ...")
    processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID_OR_PATH)
    print("Model and processor loaded.")
    return model, processor


model, processor = load_model_and_processor()

## 5. Part 1 — Attention Extraction Functions

In [ ]:
# [Define Attention Extraction Functions]
# Defines helper functions and the main extract_attentions_from_excel() function.
# No output files. These functions are called in the next cell.

def make_path(xlsx_dir: Path, base_dir_str: str, p_str: str) -> Path:
    p = Path(p_str)
    if p.is_absolute():
        return p
    if base_dir_str:
        return (Path(base_dir_str) / p).resolve()
    return (xlsx_dir / p).resolve()


def extract_attentions_from_excel(
    xlsx_path: Path,
    model,
    processor,
    base_audio_dir: str,
    base_image_dir: str,
    prompt_template: str,
    *,
    head_agg: str = "mean",
    dtype_save: str = "float16",
    audio_stride: int = 1,
    max_audio_tokens: Optional[int] = None,
    layer_stride: int = 1,
    output_dir: Optional[Path] = None,
    sheet_name: Optional[str] = None,
):
    """
    Export per-layer (Q_sub, K_sub) attention matrices (head-aggregated only)
    for each (audio, image) pair listed in the Excel file.
    """
    # --- Parameter validation ---
    head_agg = str(head_agg).lower().strip()
    assert head_agg in ("mean", "max"), f"head_agg must be 'mean' or 'max'"
    dtype_save = str(dtype_save).lower().strip()
    assert dtype_save in ("float16", "float32"), f"dtype_save must be 'float16' or 'float32'"
    np_dtype = np.float16 if dtype_save == "float16" else np.float32

    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

    # --- Output directory ---
    if output_dir is None:
        output_dir = Path(".") / "attention_pkls_raw"
    output_dir.mkdir(parents=True, exist_ok=True)

    # --- Build chat-template inputs ---
    def _build_inputs_np(image_path: Path, audio_np: np.ndarray, sr: int):
        assert isinstance(audio_np, np.ndarray) and audio_np.ndim == 1
        conversations = [
            {
                "role": "system",
                "content": [{
                    "type": "text",
                    "text": "You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of perceiving auditory and visual inputs, as well as generating text and speech."
                }],
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt_template},
                    {"type": "image", "path": str(image_path)},
                    {"type": "audio", "audio": audio_np, "sampling_rate": int(sr)},
                ],
            },
        ]
        inputs = processor.apply_chat_template(
            conversations,
            add_generation_prompt=True,
            tokenize=True,
            return_tensors="pt",
            return_dict=True,
            padding=True,
        )
        for k, v in inputs.items():
            if isinstance(v, torch.Tensor):
                inputs[k] = v.to(model.device)
        return inputs

    # --- Locate modal spans ---
    def _locate_modal_spans(inputs):
        ids = inputs["input_ids"][0].detach().cpu().tolist()
        cfg = model.config
        tok = processor.tokenizer

        def first_pos(token_id):
            if token_id is None: return None
            try: return ids.index(int(token_id))
            except ValueError: return None

        def get_token_id(attr, fallbacks):
            v = getattr(cfg, attr, None)
            if v is not None: return int(v)
            for s in fallbacks:
                try:
                    tid = tok.convert_tokens_to_ids(s)
                    if isinstance(tid, int) and tid != tok.unk_token_id:
                        return tid
                except Exception:
                    pass
            return None

        audio_start_id  = get_token_id("audio_start_token_id",  ["<|audio_start|>", "<audio>", "<|AUDIO_START|>"])
        audio_end_id    = get_token_id("audio_end_token_id",    ["<|audio_end|>", "<|AUDIO_END|>", "</audio>"])
        vision_start_id = get_token_id("vision_start_token_id", ["<|vision_start|>", "<image>", "<|image_start|>", "<|IM_START|>"])
        vision_end_id   = get_token_id("vision_end_token_id",   ["<|vision_end|>", "<image_end>", "<|image_end|>", "<|IM_END|>"])

        a_start = first_pos(audio_start_id)
        v_start = first_pos(vision_start_id)
        if a_start is None or v_start is None:
            raise RuntimeError(f"Cannot find audio/vision start tokens (audio={a_start}, vision={v_start}).")

        # Audio length
        if "audio_feature_lengths" in inputs:
            audio_len = int(inputs["audio_feature_lengths"][0].detach().cpu().item())
        elif "feature_attention_mask" in inputs:
            audio_len = int(inputs["feature_attention_mask"][0].detach().cpu().sum().item())
        elif "input_features" in inputs:
            audio_len = int(inputs["input_features"].shape[-1])
        else:
            raise RuntimeError("Cannot infer audio length.")
        a_end = a_start + audio_len

        # Image grid
        g = inputs["image_grid_thw"].detach().cpu()
        while g.ndim > 1: g = g[0]
        if g.numel() != 3:
            raise RuntimeError(f"Unexpected image_grid_thw shape: {inputs['image_grid_thw'].shape}")
        T, H_p, W_p = [int(x) for x in g.tolist()]
        grid_total = int(T * H_p * W_p)

        v_end_before_audio = None
        if vision_end_id is not None:
            try:
                cand = ids.index(int(vision_end_id), v_start + 1)
                if cand < a_start:
                    v_end_before_audio = cand
            except ValueError:
                v_end_before_audio = None

        pre_boundary_space = max(0, a_start - (v_start + 1))
        return (a_start, a_end, v_start, v_end_before_audio, (T, H_p, W_p), grid_total, pre_boundary_space)

    # --- Compose image columns (two segments) ---
    def _compose_image_columns_and_meta(
        a_start, a_end, v_start, v_end_before_audio, T, H_p, W_p,
        K_total_after_forward: int
    ):
        m_default = 2
        H_eff_def = math.ceil(H_p / m_default)
        W_eff_def = math.ceil(W_p / m_default)
        K_eff_theory_def = T * H_eff_def * W_eff_def

        post_boundary_space = max(0, K_total_after_forward - (a_end + 1))
        pre_boundary_space = max(0, a_start - (v_start + 1))
        pre_eff_len_used  = min(pre_boundary_space, K_eff_theory_def)
        post_eff_len_used = min(max(0, K_eff_theory_def - pre_eff_len_used), post_boundary_space)

        image_indices_pre  = torch.arange(v_start + 1, v_start + 1 + pre_eff_len_used, dtype=torch.long)
        post_start         = a_end + 1
        image_indices_post = torch.arange(post_start, post_start + post_eff_len_used, dtype=torch.long) if post_eff_len_used > 0 else torch.empty(0, dtype=torch.long)

        image_indices_all = torch.cat([image_indices_pre, image_indices_post], dim=0)
        K_sub = int(image_indices_all.numel())
        image_k_rel_eff = torch.arange(0, K_sub, dtype=torch.long)

        ratio = (T * H_p * W_p) / max(1, K_sub)
        m_infer = int(round(math.sqrt(ratio)))
        m_infer = max(1, min(4, m_infer))

        merge_meta = {
            "merge_size_guess": int(m_default),
            "merge_source": "assumed_default_2",
            "merge_size_inferred_from_counts": int(m_infer),
            "eff_grid_thw": (int(T), int(H_eff_def), int(W_eff_def)),
            "K_eff_theory": int(K_eff_theory_def),
            "K_sub": int(K_sub),
            "coverage_eff": float(K_sub / max(1, K_eff_theory_def))
        }
        alloc_meta = {
            "pre_boundary_space": int(pre_boundary_space),
            "post_boundary_space": int(post_boundary_space),
            "pre_eff_len_used": int(pre_eff_len_used),
            "post_eff_len_used": int(post_eff_len_used)
        }
        return (image_indices_pre, image_indices_post, image_indices_all, image_k_rel_eff,
                (T, H_eff_def, W_eff_def), merge_meta, alloc_meta)

    # --- Forward with GPU/CPU fallback ---
    def _forward_with_fallback(inputs):
        try:
            with torch.inference_mode():
                out = model(**inputs, output_attentions=True)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            return out, model.device, None
        except RuntimeError as e:
            msg = str(e)
            if any(key in msg for key in ("device-side assert", "CUDA error", "out of memory", "illegal memory access")):
                print(f"  [FALLBACK] GPU error: {msg.split(chr(10))[0]}")
                print("  -> Falling back to CPU for this sample.")
                inputs_cpu = {k: (v.detach().cpu() if isinstance(v, torch.Tensor) else v) for k, v in inputs.items()}
                model_cpu = model.to("cpu")
                with torch.inference_mode():
                    out_cpu = model_cpu(**inputs_cpu, output_attentions=True)
                model_cpu.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                return out_cpu, torch.device("cpu"), f"forward:{msg.splitlines()[0]}"
            else:
                raise

    # ==================== Main processing loop ====================
    print(f"\n>>> Processing: {xlsx_path.name}")
    try:
        df = pd.read_excel(xlsx_path, engine="openpyxl") if sheet_name is None else pd.read_excel(xlsx_path, sheet_name=sheet_name, engine="openpyxl")
    except Exception as e:
        print(f"  [ERROR] Cannot read Excel: {e}")
        return

    for c in ["Item", "Audio_File_new", "Image_File"]:
        if c not in df.columns:
            print(f"  [ERROR] {xlsx_path.name} missing column: {c}. Skipping.")
            return

    target_sr = processor.feature_extractor.sampling_rate
    print(f"    Target sampling rate: {target_sr} Hz")
    resampler_cache: Dict[int, torchaudio.transforms.Resample] = {}
    xlsx_dir = xlsx_path.parent

    model_id = (
        str(getattr(model.config, "_name_or_path", "")) or
        str(getattr(model, "name_or_path", "")) or
        MODEL_ID_OR_PATH
    )

    for _, row in df.iterrows():
        item = row["Item"]
        try:
            audio_path = make_path(xlsx_dir, base_audio_dir, str(row["Audio_File_new"]).strip())
            image_path = make_path(xlsx_dir, base_image_dir, str(row["Image_File"]).strip())
        except Exception as e:
            print(f"  [SKIP] Item={item} path error: {e}"); continue
        if not audio_path.exists():
            print(f"  [SKIP] Item={item} audio not found: {audio_path}"); continue
        if not image_path.exists():
            print(f"  [SKIP] Item={item} image not found: {image_path}"); continue

        safe_item = str(item).replace('/', '_').replace('\\', '_')
        pkl_name = f"Item_{safe_item}__{audio_path.stem}__{image_path.stem}.pkl"
        pkl_path = output_dir / pkl_name
        if pkl_path.exists():
            print(f"  [SKIP] Item={item} PKL already exists: {pkl_name}"); continue

        print(f"  --- Processing Item={item} ---")
        try:
            # Load audio
            try:
                audio_wav, sr0 = torchaudio.load(audio_path)
            except Exception as e:
                print(f"  [SKIP] Item={item} cannot load audio: {e}"); continue
            if sr0 != target_sr:
                if sr0 not in resampler_cache:
                    resampler_cache[sr0] = torchaudio.transforms.Resample(sr0, target_sr).to(audio_wav.device)
                audio_wav = resampler_cache[sr0](audio_wav)
            if audio_wav.shape[0] > 1: audio_wav = audio_wav.mean(dim=0, keepdim=True)
            audio_np = audio_wav.squeeze(0).detach().cpu().numpy().astype(np.float32, copy=False)

            # Verify image
            try:
                _ = Image.open(image_path).convert("RGB")
            except Exception as e:
                print(f"  [SKIP] Item={item} cannot load image: {e}"); continue

            # Build inputs
            inputs = _build_inputs_np(image_path=image_path, audio_np=audio_np, sr=target_sr)

            # Locate modal spans
            try:
                a_start, a_end, v_start, v_end_before_audio, (T, H_p, W_p), grid_total, pre_boundary_space = _locate_modal_spans(inputs)
            except Exception as e:
                print(f"  [SKIP] Item={item} cannot locate modal spans: {e}"); continue

            # Forward pass
            out, attn_device, forward_fallback = _forward_with_fallback(inputs)
            attentions = out.attentions
            if attentions is None:
                raise RuntimeError("Model did not return attentions; ensure attn_implementation='eager'.")

            H_heads, Q_total, K_total = attentions[0].squeeze(0).shape

            # Compose image columns
            (image_idx_pre, image_idx_post, image_idx_all,
             image_k_rel_eff, eff_grid_thw, merge_meta, alloc_meta) = _compose_image_columns_and_meta(
                a_start, a_end, v_start, v_end_before_audio, T, H_p, W_p, K_total_after_forward=K_total
            )

            # Clip to K_total range
            ok_mask = (image_idx_all >= 0) & (image_idx_all < K_total)
            image_idx_all = image_idx_all[ok_mask]
            image_k_rel_eff = image_k_rel_eff[ok_mask]
            if image_idx_pre.numel() > 0:
                image_idx_pre = image_idx_pre[(image_idx_pre >= 0) & (image_idx_pre < K_total)]
            if image_idx_post.numel() > 0:
                image_idx_post = image_idx_post[(image_idx_post >= 0) & (image_idx_post < K_total)]

            # Audio Q segment
            audio_idx_cpu = torch.arange(a_start + 1, a_start + 1 + (a_end - a_start), dtype=torch.long)
            audio_idx_cpu = audio_idx_cpu[(audio_idx_cpu >= 0) & (audio_idx_cpu < Q_total)]
            if audio_stride > 1:
                audio_idx_cpu = audio_idx_cpu[::int(audio_stride)]
            if max_audio_tokens is not None and audio_idx_cpu.numel() > int(max_audio_tokens):
                audio_idx_cpu = audio_idx_cpu[:int(max_audio_tokens)]
            if audio_idx_cpu.numel() == 0 or image_idx_all.numel() == 0:
                print(f"  [SKIP] Item={item} empty indices: Q_total={Q_total}, K_total={K_total}"); continue

            Q_sub = int(audio_idx_cpu.numel()); K_sub = int(image_idx_all.numel())
            print(f"    layers={len(attentions)} (using {len(range(0, len(attentions), int(max(1, layer_stride))))}), Q_sub={Q_sub}, K_sub={K_sub}, H={H_heads}")

            device = attn_device if torch.cuda.is_available() else torch.device("cpu")
            use_cpu_post = (device.type == "cpu")
            audio_rows_gpu = audio_idx_cpu.to(device)
            image_cols_gpu = image_idx_all.to(device)

            # Extract per-layer attention
            layer_ids = list(range(0, len(attentions), int(max(1, layer_stride))))
            attn_mats: List[np.ndarray] = []
            post_fallback_reason = None

            for li in layer_ids:
                L_bhqk = attentions[li][0]
                if not use_cpu_post:
                    try:
                        sub = L_bhqk.index_select(1, audio_rows_gpu).index_select(2, image_cols_gpu)
                        M = sub.mean(dim=0) if head_agg == "mean" else sub.amax(dim=0)
                        arr = M.detach().to("cpu", dtype=torch.float32).numpy().astype(np_dtype, copy=False)
                        attn_mats.append(arr)
                        del sub, M
                        if torch.cuda.is_available(): torch.cuda.empty_cache()
                        continue
                    except RuntimeError as e:
                        post_fallback_reason = f"postproc:{str(e).splitlines()[0]}"
                        use_cpu_post = True

                layer_cpu = L_bhqk.detach().to("cpu", dtype=torch.float32)
                sub_cpu = layer_cpu.index_select(1, audio_idx_cpu).index_select(2, image_idx_all)
                M_cpu = sub_cpu.mean(dim=0) if head_agg == "mean" else sub_cpu.amax(dim=0)
                attn_mats.append(M_cpu.numpy().astype(np_dtype, copy=False))
                del layer_cpu, sub_cpu, M_cpu

            # Effective 2D coordinates
            T_eff, H_eff, W_eff = eff_grid_thw
            k_rel_eff_np = image_k_rel_eff.detach().cpu().numpy()
            t_eff = (k_rel_eff_np // (H_eff * W_eff)).astype(np.int32) if T_eff > 1 else np.zeros_like(k_rel_eff_np, dtype=np.int32)
            hw_rem = k_rel_eff_np % (H_eff * W_eff)
            h_eff = (hw_rem // W_eff).astype(np.int32)
            w_eff = (hw_rem %  W_eff).astype(np.int32)
            eff_coords = np.stack([t_eff, h_eff, w_eff], axis=1)

            # Assemble payload
            payload = {
                "schema_version": "qk_attn.v2",
                "model_id": model_id,
                "head_agg": head_agg,
                "dtype": dtype_save,
                "item": item,
                "audio_path": str(audio_path),
                "image_path": str(image_path),
                "Q_total": int(Q_total),
                "K_total": int(K_total),
                "audio_indices": audio_idx_cpu.detach().cpu().tolist(),
                "image_indices_pre": image_idx_pre.detach().cpu().tolist(),
                "image_indices_post": image_idx_post.detach().cpu().tolist(),
                "image_indices": image_idx_all.detach().cpu().tolist(),
                "image_grid_thw": (int(T), int(H_p), int(W_p)),
                "eff_grid_thw": (int(T_eff), int(H_eff), int(W_eff)),
                "image_k_rel_eff": image_k_rel_eff.detach().cpu().tolist(),
                "eff_coords_per_token": eff_coords.tolist(),
                "layer_ids": [int(x) for x in layer_ids],
                "attn_per_layer": attn_mats,
                "notes": {
                    "device_forward": str(attn_device),
                    "device_postproc": "cpu" if (use_cpu_post or attn_device.type == "cpu") else "cuda",
                    "forward_fallback": forward_fallback,
                    "postproc_fallback": post_fallback_reason,
                    "H_heads": int(H_heads),
                    "AUDIO_STRIDE": int(audio_stride),
                    "MAX_AUDIO_TOKENS": (None if max_audio_tokens is None else int(max_audio_tokens)),
                    "LAYER_STRIDE": int(layer_stride),
                    "a_start": int(a_start), "a_end": int(a_end),
                    "v_start": int(v_start),
                    "v_end_before_audio": (None if v_end_before_audio is None else int(v_end_before_audio)),
                    "grid_total": int(grid_total),
                    **merge_meta,
                    **alloc_meta,
                }
            }

            with open(pkl_path, "wb") as f:
                pickle.dump(payload, f, protocol=4)
            print(f"  [+] Saved: {pkl_name}")

        except Exception as e:
            print(f"  [FAIL] Item={item}: {e}")

    print(f"\n<<< Finished: {xlsx_path.name}")


print("Attention extraction functions defined.")

## 6. Run Attention Extraction

Uses the selected `AUDIO_MODE` to determine audio directory and PKL output path.

In [ ]:
# [Run Attention Extraction]
# Reads all .xlsx files in ROOT_DIR, processes each (audio, image) pair
# through the model, and saves per-layer attention matrices.
# Output files: one .pkl file per item, saved to {ROOT_DIR}/{pkl_subdir}/
# Skips items whose .pkl already exists (delete old .pkl to re-extract).

xlsx_files = sorted([p for p in ROOT_DIR.glob("*.xlsx") if not p.name.startswith("~$")])

if not xlsx_files:
    print(f"No .xlsx files found in {ROOT_DIR.resolve()}")
else:
    audio_dir = ACTIVE_CONDITION["audio_dir"]
    pkl_out = ROOT_DIR / ACTIVE_CONDITION["pkl_subdir"]
    pkl_out.mkdir(parents=True, exist_ok=True)

    print(f"AUDIO_MODE:  {AUDIO_MODE}")
    print(f"Audio dir:   {audio_dir}")
    print(f"PKL output:  {pkl_out}")
    print(f"Excel files: {len(xlsx_files)}")
    print("=" * 60)

    for x_file in xlsx_files:
        extract_attentions_from_excel(
            xlsx_path=x_file,
            model=model,
            processor=processor,
            base_audio_dir=audio_dir,
            base_image_dir=BASE_IMAGE_DIR_STR,
            prompt_template=PROMPT_TEMPLATE,
            head_agg=HEAD_AGG,
            dtype_save=DTYPE_SAVE,
            audio_stride=AUDIO_STRIDE,
            max_audio_tokens=MAX_AUDIO_TOKENS,
            layer_stride=LAYER_STRIDE,
            output_dir=pkl_out,
            sheet_name=SHEET_NAME,
        )
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("\n=== Attention extraction complete ===")

---
## 7. Part 2 — Region Analysis Functions

Reads the `.pkl` files from Part 1, maps attention to four image quadrants (TL/TR/BL/BR), and exports normalized CSV files.

In [ ]:
# [Define Region Analysis Functions]
# Defines functions to map image tokens to quadrant regions (TL/TR/BL/BR),
# aggregate attention per region, normalize, and export to CSV.
# No output files. These functions are called in the next cell.

# Fixed box constants (derived from REGIONS and CANVAS settings)
CELL_PX = PATCH_SIZE * 2  # effective cell pixel size (= 14*2 = 28)
FIXED_BOXES = REGIONS.copy()

H_EFF_EXPECT = CANVAS_H // CELL_PX
W_EFF_EXPECT = CANVAS_W // CELL_PX
TOKENS_PER_BOX_EXPECT = (
    (FIXED_BOXES["TL"][2] - FIXED_BOXES["TL"][0]) // CELL_PX
) * (
    (FIXED_BOXES["TL"][3] - FIXED_BOXES["TL"][1]) // CELL_PX
)
TOTAL_BOX_TOKENS_EXPECT = TOKENS_PER_BOX_EXPECT * 4

print(f"Effective grid: {H_EFF_EXPECT} x {W_EFF_EXPECT} (cell={CELL_PX}px)")
print(f"Tokens per box: {TOKENS_PER_BOX_EXPECT}, total (4 boxes): {TOTAL_BOX_TOKENS_EXPECT}")


def _load_v2_strict(pkl_path: str) -> dict:
    d = pickle.load(open(pkl_path, "rb"))
    if d.get("schema_version", "") != "qk_attn.v2":
        raise RuntimeError(f"{pkl_path}: requires schema_version='qk_attn.v2', got {d.get('schema_version')}")
    if "eff_grid_thw" not in d or "image_k_rel_eff" not in d or "eff_coords_per_token" not in d:
        raise RuntimeError(f"{pkl_path}: missing v2 fields eff_grid_thw/image_k_rel_eff/eff_coords_per_token")
    return d


def _assert_canvas_and_grid(d: dict) -> Tuple[int, int, Tuple[int, int, int]]:
    W0, H0 = Image.open(d["image_path"]).size
    if (W0, H0) != (CANVAS_W, CANVAS_H):
        raise RuntimeError(f"Image resolution mismatch: expected {CANVAS_W}x{CANVAS_H}, got {W0}x{H0}")
    T_eff, H_eff, W_eff = d["eff_grid_thw"]
    if not (T_eff == 1 and H_eff == H_EFF_EXPECT and W_eff == W_EFF_EXPECT):
        raise RuntimeError(f"Grid mismatch: expected (1,{H_EFF_EXPECT},{W_EFF_EXPECT}), got ({T_eff},{H_eff},{W_eff})")
    if W0 // W_eff != CELL_PX or H0 // H_eff != CELL_PX:
        raise RuntimeError(f"Cell size mismatch: expected {CELL_PX}px")
    return W0, H0, (T_eff, H_eff, W_eff)


def _boxes_to_eff_ranges_strict(W_eff: int, H_eff: int) -> Dict[str, Tuple[range, range]]:
    ranges: Dict[str, Tuple[range, range]] = {}
    for name, (x1, y1, x2, y2) in FIXED_BOXES.items():
        for v in (x1, y1, x2, y2):
            if v % CELL_PX != 0:
                raise RuntimeError(f"Box {name} boundary {v} is not a multiple of {CELL_PX}")
        gx1 = x1 // CELL_PX; gx2 = x2 // CELL_PX
        gy1 = y1 // CELL_PX; gy2 = y2 // CELL_PX
        if not (0 <= gx1 < gx2 <= W_eff and 0 <= gy1 < gy2 <= H_eff):
            raise RuntimeError(f"Box {name} out of grid range")
        cols = gx2 - gx1; rows = gy2 - gy1
        if rows * cols != TOKENS_PER_BOX_EXPECT:
            raise RuntimeError(f"Box {name} token count: {rows}x{cols}={rows*cols}, expected {TOKENS_PER_BOX_EXPECT}")
        ranges[name] = (range(gy1, gy2), range(gx1, gx2))
    # Check no overlap
    seen = set()
    for name, (rows, cols) in ranges.items():
        for y in rows:
            for x in cols:
                if (y, x) in seen:
                    raise RuntimeError(f"Overlapping cell at ({y},{x})")
                seen.add((y, x))
    return ranges


def _ranges_to_krel(rows: range, cols: range, W_eff: int) -> np.ndarray:
    return np.asarray([y * W_eff + x for y in rows for x in cols], dtype=np.int64)


def _build_region_cols_from_pkl(d: dict, include_rest: bool = False) -> Dict[str, np.ndarray]:
    _, _, (T_eff, H_eff, W_eff) = _assert_canvas_and_grid(d)
    if T_eff != 1:
        raise RuntimeError(f"Only static images supported (T_eff=1), got T_eff={T_eff}")
    ranges = _boxes_to_eff_ranges_strict(W_eff=W_eff, H_eff=H_eff)
    k_rel_have = np.asarray(d["image_k_rel_eff"], dtype=np.int64)
    K_sub = int(k_rel_have.shape[0])
    idx_map: Dict[int, int] = {}
    for col, k in enumerate(k_rel_have.tolist()):
        if k in idx_map:
            raise RuntimeError(f"Duplicate k_rel_eff: k={k}")
        idx_map[k] = col

    region_cols: Dict[str, np.ndarray] = {}
    all_cols_list = []
    for name, (rows, cols) in ranges.items():
        need_k = _ranges_to_krel(rows, cols, W_eff)
        miss = [int(k) for k in need_k.tolist() if k not in idx_map]
        if miss:
            raise RuntimeError(f"PKL missing {len(miss)} tokens for box {name}")
        cols_in_sub = np.asarray([idx_map[int(k)] for k in need_k.tolist()], dtype=np.int64)
        if cols_in_sub.size != TOKENS_PER_BOX_EXPECT:
            raise RuntimeError(f"Box {name} column count: {cols_in_sub.size}, expected {TOKENS_PER_BOX_EXPECT}")
        region_cols[name] = cols_in_sub
        all_cols_list.append(cols_in_sub)

    all_cols_cat = np.concatenate(all_cols_list, axis=0)
    if all_cols_cat.size != TOTAL_BOX_TOKENS_EXPECT:
        raise RuntimeError(f"Total box columns: {all_cols_cat.size}, expected {TOTAL_BOX_TOKENS_EXPECT}")
    unique_cols = np.unique(all_cols_cat)
    if unique_cols.size != TOTAL_BOX_TOKENS_EXPECT:
        raise RuntimeError(f"Overlapping columns detected")

    if include_rest:
        full = np.arange(K_sub, dtype=np.int64)
        region_cols["REST"] = np.setdiff1d(full, unique_cols, assume_unique=False)

    return region_cols


def _aggregate_layers_and_normalize(
    d: dict,
    region_cols: Dict[str, np.ndarray],
    agg: str = "mean",
    decimals: int = 6
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    attn_layers = d["attn_per_layer"]
    layer_ids = d.get("layer_ids", list(range(len(attn_layers))))
    audio_abs = np.asarray(d["audio_indices"], dtype=np.int64)
    Q_sub, K_sub = attn_layers[0].shape

    order = ["TL", "TR", "BL", "BR"] + (["REST"] if "REST" in region_cols else [])
    R = len(order)

    def _agg_one(A):
        A = A.astype(np.float32, copy=False)
        out = np.zeros((Q_sub, R), dtype=np.float32)
        for j, name in enumerate(order):
            cols = region_cols[name]
            if cols.size == 0:
                out[:, j] = 0.0; continue
            sub = A[:, cols]
            if agg == "sum":    v = sub.sum(axis=1)
            elif agg == "max":  v = sub.max(axis=1)
            else:               v = sub.mean(axis=1)
            out[:, j] = v
        rs = out.sum(axis=1, keepdims=True)
        zero_mask = (rs == 0.0)
        rs[zero_mask] = 1.0
        out = out / rs
        if np.any(zero_mask):
            out[zero_mask.flatten(), :] = 1.0 / float(R)
        return out

    per_layer = [_agg_one(A) for A in attn_layers]
    layer_labels = [f"L{lid+1:02d}" for lid in layer_ids]
    q_idx = np.arange(Q_sub, dtype=int)

    # Human-readable
    data_h = {"q_idx": q_idx.tolist(), "audio_abs_index": audio_abs.tolist()}
    for mat, lab in zip(per_layer, layer_labels):
        rounded = np.round(mat, decimals)
        col_str = []
        for i in range(rounded.shape[0]):
            parts = [f"{name}={rounded[i, j]:.{decimals}f}" for j, name in enumerate(order)]
            col_str.append(";".join(parts))
        data_h[lab] = np.array(col_str, dtype=object)
    df_human = pd.DataFrame(data_h)

    # Machine-friendly
    data_m = {"q_idx": q_idx.tolist(), "audio_abs_index": audio_abs.tolist()}
    for mat, lab in zip(per_layer, layer_labels):
        out = np.round(mat, decimals=decimals)
        for j, name in enumerate(order):
            data_m[f"{lab}_{name}"] = out[:, j]
    df_machine = pd.DataFrame(data_m)

    return df_human, df_machine


def process_one_pkl_fixed_boxes(
    pkl_path: Path,
    out_root: Path,
    agg: str = "mean",
    decimals: int = 6,
    include_rest: bool = False
) -> Tuple[Path, Path]:
    d = _load_v2_strict(str(pkl_path))
    region_cols = _build_region_cols_from_pkl(d, include_rest=include_rest)

    sizes = {k: int(len(v)) for k, v in region_cols.items()}
    total_known = sizes.get("TL", 0) + sizes.get("TR", 0) + sizes.get("BL", 0) + sizes.get("BR", 0)
    rest_sz = sizes.get("REST", 0)
    K_sub = len(d["image_k_rel_eff"])
    print(f"[{pkl_path.name}] TL/TR/BL/BR={total_known}, REST={rest_sz}, total={total_known + rest_sz}/{K_sub}")

    df_human, df_machine = _aggregate_layers_and_normalize(d, region_cols, agg=agg, decimals=decimals)

    out_human_dir = out_root / "human_readable"
    out_machine_dir = out_root / "machine_friendly"
    out_human_dir.mkdir(parents=True, exist_ok=True)
    out_machine_dir.mkdir(parents=True, exist_ok=True)

    stem = pkl_path.stem
    tag = "fixedboxes5" if ("REST" in region_cols) else "fixedboxes4"
    human_csv = out_human_dir / f"{stem}.{tag}_{agg}_norm.human.csv"
    machine_csv = out_machine_dir / f"{stem}.{tag}_{agg}_norm.machine.csv"

    df_human.to_csv(human_csv, index=False, encoding="utf-8")
    df_machine.to_csv(machine_csv, index=False, encoding="utf-8")
    print(f"  -> HUMAN:   {human_csv}")
    print(f"  -> MACHINE: {machine_csv}")
    return human_csv, machine_csv


def process_dir_fixed_boxes(
    pkl_dir: Path,
    out_root: Path,
    pattern: str = "*.pkl",
    agg: str = "mean",
    decimals: int = 6,
    fail_fast: bool = True,
    include_rest: bool = False
) -> None:
    pkl_files = sorted(pkl_dir.glob(pattern))
    if not pkl_files:
        print(f"[WARN] No PKL found in {pkl_dir} (pattern={pattern})")
        return
    print(f"[INFO] Found {len(pkl_files)} PKL in {pkl_dir}")
    out_root.mkdir(parents=True, exist_ok=True)

    ok, fail = 0, 0
    for p in pkl_files:
        try:
            process_one_pkl_fixed_boxes(p, out_root=out_root, agg=agg, decimals=decimals, include_rest=include_rest)
            ok += 1
        except Exception as e:
            print(f"[FAIL] {p.name}: {e}")
            fail += 1
            if fail_fast:
                raise
    print(f"[DONE] success={ok}, failed={fail}")


print("Region analysis functions defined.")

## 8. Run Region Analysis

Uses the same `AUDIO_MODE` to locate PKL files and export CSV results.

In [ ]:
# [Run Region Analysis]
# Reads .pkl files from the extraction step, maps attention to four quadrants,
# and exports normalized results as CSV.
# Output files (saved to {ROOT_DIR}/{csv_subdir}/):
#   - human_readable/*.human.csv   (one string column per layer)
#   - machine_friendly/*.machine.csv (separate numeric columns per layer per region)

pkl_dir = ROOT_DIR / ACTIVE_CONDITION["pkl_subdir"]
csv_out = ROOT_DIR / ACTIVE_CONDITION["csv_subdir"]

print(f"AUDIO_MODE:  {AUDIO_MODE}")
print(f"PKL input:   {pkl_dir}")
print(f"CSV output:  {csv_out}")
print("=" * 60)

process_dir_fixed_boxes(
    pkl_dir=pkl_dir,
    out_root=csv_out,
    pattern="*.pkl",
    agg=REGION_AGG,
    decimals=DECIMALS,
    fail_fast=FAIL_FAST,
    include_rest=INCLUDE_REST,
)

print("\n=== Region analysis complete ===")